# Plotting with PyQTgraph

Before starting, please install *PyQT Graph* tools
```bash
    > pip install pyqt6 pyqtgraph
``` 

## Plotting signals from remote antenna

In [8]:
# Load packages

import numpy as np
import queue
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros.core.ws import MemsArrayWS
import pyqtgraph as pg
from pyqtgraph.Qt import QtCore

log.setLevel( "INFO" )

# Set server access credentials
#HOST = 'buzenval20.fr'
HOST = 'parisparc.biimea.tech'
#HOST = 'localhost'
PORT = 9002
MEMS = [0, 1, 2, 3, 4, 5]

## Connecting to the remote server

Providing a *MBS* server is running at ``HOST:PORT``, one can try to connect by creating a ``MemsArrayWS`` object.

In [ ]:
# Define the antenna
try:
    antenna = MemsArrayWS( HOST, port=PORT )
except Exception as e:
    print( f"Failed: {e}" )

2023-11-23 09:32:05,563 [INFO]:  .Install MemsArrayWS settings
2023-11-23 09:32:05,578 [INFO]:  .Created a new antenna
2023-11-23 09:32:05,580 [INFO]:  .Async event loop already running. Adding coroutine to the event loop...


2023-11-23 09:32:05,588 [INFO]:  .Try connecting to ws://parisparc.biimea.tech:9002...
2023-11-23 09:32:05,891 [INFO]:  .Received positive answer from server
2023-11-23 09:32:05,892 [INFO]:  .Getting settings values from remote receiver...
2023-11-23 09:32:05,989 [INFO]:  .Received settings from server [ok]
2023-11-23 09:32:05,990 [INFO]:  .Set 32 available MEMs numbered from 0 to 31
2023-11-23 09:32:05,991 [INFO]:  .No analogic channels available
2023-11-23 09:32:06,073 [INFO]:  .Starting MegamicrosWS device [ready]


## Init graphic PyQT library

In [9]:
# init PyQtgraph
win = pg.GraphicsLayoutWidget(show=True, title="Plotting database signals")
win.resize(1000,600)
win.setWindowTitle('Plotting database signals')
pg.setConfigOptions(antialias=True)
graph = win.addPlot(title="Microphones")
graph.setYRange(-5,5, padding=0, update = False)
curves = []
for s in range( len( MEMS ) ):
    curves.append( graph.plot(pen='y' ) )

# Define the plot function
# Get last queued signal and plot it
def plot_on_the_fly( antenna, curves ):

	try:
		data = antenna.signal_q.get( timeout=1 )
	except queue.Empty:
		return

	t = np.arange( np.size( data, 1 ) )/antenna.sampling_frequency
	for s in range( antenna.mems_number ):
		curves[s].setData( t, ( data[s,:] * antenna.sensibility ) + s - antenna.mems_number/2 )


timer = QtCore.QTimer()
timer.timeout.connect( lambda: plot_on_the_fly( antenna, curves ) )

## Plotting signals from DbIA database

One has only to specify the file identifier.

The ``MemsArrayDB`` constructor connects to the database and populates the antenna parameters with metadata received from the database. An exception is raised if connection failed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import queue
from megamicros.log import log
from megamicros.core.db import MemsArrayDB

log.setLevel( "INFO" )

# Set database access credentials
#DBHOST = 'http://dbwelfare.biimea.io/'
DBHOST = 'http://dbchantier.biimea.io/'
LOGIN = 'ailab'
EMAIL = 'bruno.gas@biimea.com'
PASSWORD = '#T;uZnQ5UJ_JC~&'

In [ ]:
# Choose the a signal file of the database and the MEMs to plot
FILE_ID = 1239
FILE_ID = 1231
MEMS = [0, 1, 2, 3, 4, 5, 6],

# Define the antenna
antenna = MemsArrayDB( 
    dbhost=DBHOST, login=LOGIN, email=EMAIL, password=PASSWORD, 
    file_id=FILE_ID
)

## Init graph

Get data as `float32` type. 

In [ ]:
# init PyQtgraph
win = pg.GraphicsLayoutWidget(show=True, title="Plotting database signals")
win.resize(1000,600)
win.setWindowTitle('Plotting database signals')
pg.setConfigOptions(antialias=True)
graph = win.addPlot(title="Microphones")
graph.setYRange(-5,5, padding=0, update = False)
curves = []
for s in range( len( MEMS ) ):
    curves.append( graph.plot(pen='y' ) )

# Define the plot function
# Get last queued signal and plot it
def plot_on_the_fly( antenna, curves ):

	try:
		data = antenna.signal_q.get( timeout=1 )
	except queue.Empty:
		return
	
	print( f"ok!" )

	t = np.arange( np.size( data, 1 ) )/antenna.sampling_frequency
	for s in range( antenna.mems_number ):
		curves[s].setData( t, ( data[s,:] * antenna.sensibility ) + s - antenna.mems_number/2 )


timer = QtCore.QTimer()
timer.timeout.connect( lambda: plot_on_the_fly( antenna, curves ) )

## Start acquisition and plotting

In [10]:
antenna.run( 
    mems=MEMS,									# activated mems
    duration=4,
    sampling_frequency=20000,					# sampling frequency
    buffer_length=512,
    signal_q_size=0,
    counter = False
)

# Start and set the timer period in milliseconds
timer.start( 1 )

input( "Press [Return] key to stop...\n" )

timer.stop()
antenna.wait()

2023-11-23 09:45:12,347 [INFO]:  .Starting run execution
2023-11-23 09:45:12,362 [INFO]:  .Install MemsArray settings
2023-11-23 09:45:12,364 [INFO]:  .6 MEMs were activated among 0 to 31 available MEMs
2023-11-23 09:45:12,365 [INFO]:  .Install MemsArrayWS settings
2023-11-23 09:45:12,366 [INFO]:  .Pre-execution checks for MemsArray.run()
2023-11-23 09:45:12,367 [INFO]:  .Counter skipping not set -> set to False
2023-11-23 09:45:12,368 [INFO]:  .Status was not set -> set to False
2023-11-23 09:45:12,370 [INFO]:  .Requested job: run
2023-11-23 09:45:12,371 [INFO]:  .Perform a 4s run loop
2023-11-23 09:45:12,372 [INFO]:  .H5 recording off
2023-11-23 09:45:12,373 [INFO]:  .Start a `run` running job on remote server
2023-11-23 09:45:12,374 [INFO]:  .Background execution mode off
2023-11-23 09:45:12,375 [INFO]:  .Run thread execution started
2023-11-23 09:45:12,377 [INFO]:  .Connecting to remote host parisparc.biimea.tech:9002...
2023-11-23 09:45:12,826 [INFO]:  .Connected
2023-11-23 09:45:

In [ ]:
# Run antenna
antenna.run(
    mems = MEMS,
    start = 120,
    duration=10,
    counter_skip = True,
    datatype='float32',
)

# Init a np.ndarray
signals = np.ndarray( (0, antenna.channels_number ) )

# Get signals
for data in antenna:
    signals = np.concatenate( ( signals, data ), axis=0 )

# waiting for the end of the running thread is mandatory
antenna.wait()
print( f"exit from loop. Signal shape is: {np.shape( signals )}" )

In [ ]:
# halt server
antenna.shutdown()